# generic

> `GenericGraphStorageAdapter` — the generic (backend-agnostic) implementation of the graph-storage task contract. Thread-3 packaging: the GENERIC adapter impl is the dominant case and lives in the dep-light interface lib, reused across all compatible backend tools. Its job is **wire-input normalization + forwarding**: accept wire dicts or typed objects at the task boundary, hand the tool purely-typed arguments, pass typed results through (they wire-encode at the worker boundary).

In [ ]:
#| default_exp generic

In [ ]:
#| export
from typing import Any, Dict, List, Optional

from cjm_context_graph_primitives.provenance import SourceRef
from cjm_context_graph_primitives.graph import GraphNode, GraphEdge, GraphContext
from cjm_context_graph_primitives.query import (
    NodeQuery, EdgeQuery, RawQuery,
    NodeQueryResult, EdgeQueryResult, RawQueryResult,
)

from cjm_graph_storage_adapter_interface.adapter import GraphStorageAdapter

In [ ]:
#| export
def ensure_node(
    value: Any,  # GraphNode or its wire dict
) -> GraphNode:  # Typed node
    """Normalize a wire dict to a `GraphNode` (typed values pass through)."""
    return value if isinstance(value, GraphNode) else GraphNode.from_dict(value)

In [ ]:
#| export
def ensure_edge(
    value: Any,  # GraphEdge or its wire dict
) -> GraphEdge:  # Typed edge
    """Normalize a wire dict to a `GraphEdge` (typed values pass through)."""
    return value if isinstance(value, GraphEdge) else GraphEdge.from_dict(value)

In [ ]:
#| export
def ensure_context(
    value: Any,  # GraphContext or its wire dict
) -> GraphContext:  # Typed context
    """Normalize a wire dict to a `GraphContext` (typed values pass through)."""
    return value if isinstance(value, GraphContext) else GraphContext.from_dict(value)

In [ ]:
#| export
def ensure_source_ref(
    value: Any,  # SourceRef or its wire dict
) -> SourceRef:  # Typed source ref
    """Normalize a wire dict to a `SourceRef` (typed values pass through)."""
    return value if isinstance(value, SourceRef) else SourceRef.from_dict(value)

In [ ]:
#| export
def ensure_node_query(
    value: Any,  # NodeQuery or its tagged wire dict
) -> NodeQuery:  # Typed query
    """Normalize a tagged wire dict to a `NodeQuery` (typed values pass through)."""
    return value if isinstance(value, NodeQuery) else NodeQuery.from_dict(value)

In [ ]:
#| export
def ensure_edge_query(
    value: Any,  # EdgeQuery or its tagged wire dict
) -> EdgeQuery:  # Typed query
    """Normalize a tagged wire dict to an `EdgeQuery` (typed values pass through)."""
    return value if isinstance(value, EdgeQuery) else EdgeQuery.from_dict(value)

In [ ]:
#| export
def ensure_raw_query(
    value: Any,  # RawQuery or its tagged wire dict
) -> RawQuery:  # Typed query
    """Normalize a tagged wire dict to a `RawQuery` (typed values pass through)."""
    return value if isinstance(value, RawQuery) else RawQuery.from_dict(value)

In [ ]:
#| export
class GenericGraphStorageAdapter(GraphStorageAdapter):
    """Generic graph-storage adapter: normalize wire inputs, forward to the
    bound tool, pass typed results through.

    Works against ANY tool satisfying `GraphStorageToolProtocol` — the
    backend owns translation, so nothing here is backend-specific.
    """

    def add_nodes(self, nodes: List[Any]) -> List[str]:
        """Bulk-create nodes."""
        return self.tool.add_nodes([ensure_node(n) for n in nodes])

    def add_edges(self, edges: List[Any]) -> List[str]:
        """Bulk-create edges."""
        return self.tool.add_edges([ensure_edge(e) for e in edges])

    def get_node(self, node_id: str) -> Optional[GraphNode]:
        """Fetch a single node by id."""
        return self.tool.get_node(node_id)

    def get_edge(self, edge_id: str) -> Optional[GraphEdge]:
        """Fetch a single edge by id."""
        return self.tool.get_edge(edge_id)

    def get_context(self, node_id: str, depth: int = 1,
                    filter_labels: Optional[List[str]] = None) -> GraphContext:
        """Fetch a node's neighborhood subgraph."""
        return self.tool.get_context(node_id, depth=depth, filter_labels=filter_labels)

    def find_nodes_by_source(self, source_ref: Any) -> List[GraphNode]:
        """Reverse provenance lookup (content-hash-primary)."""
        return self.tool.find_nodes_by_source(ensure_source_ref(source_ref))

    def find_nodes_by_label(self, label: str, limit: int = 100) -> List[GraphNode]:
        """Fetch nodes by label."""
        return self.tool.find_nodes_by_label(label, limit=limit)

    def query_nodes(self, query: Any) -> NodeQueryResult:
        """Execute a typed node query."""
        return self.tool.query_nodes(ensure_node_query(query))

    def query_edges(self, query: Any) -> EdgeQueryResult:
        """Execute a typed edge query."""
        return self.tool.query_edges(ensure_edge_query(query))

    def raw_query(self, query: Any) -> RawQueryResult:
        """Execute the marked, backend-coupled raw escape."""
        return self.tool.raw_query(ensure_raw_query(query))

    def update_node(self, node_id: str, properties: Dict[str, Any]) -> bool:
        """Merge properties into a node."""
        return self.tool.update_node(node_id, properties)

    def update_edge(self, edge_id: str, properties: Dict[str, Any]) -> bool:
        """Merge properties into an edge."""
        return self.tool.update_edge(edge_id, properties)

    def delete_nodes(self, node_ids: List[str], cascade: bool = True) -> int:
        """Bulk-delete nodes."""
        return self.tool.delete_nodes(node_ids, cascade=cascade)

    def delete_edges(self, edge_ids: List[str]) -> int:
        """Bulk-delete edges."""
        return self.tool.delete_edges(edge_ids)

    def get_schema(self) -> Dict[str, Any]:
        """Report the stored graph's schema."""
        return self.tool.get_schema()

    def integrity_check(self) -> Dict[str, Any]:
        """Backend self-check (G3 institutionalized)."""
        return self.tool.integrity_check()

    def import_graph(self, graph_data: Any, merge_strategy: str = "overwrite") -> Dict[str, int]:
        """Bulk-import a subgraph."""
        return self.tool.import_graph(ensure_context(graph_data), merge_strategy=merge_strategy)

    def export_graph(self, filter_query: Optional[Any] = None) -> GraphContext:
        """Export the graph (optionally filtered by a typed node query)."""
        fq = ensure_node_query(filter_query) if filter_query is not None else None
        return self.tool.export_graph(fq)

In [ ]:
# Contract-level test: an in-memory fake tool satisfying the protocol
# (per the contract-level-reconfigure-tests discipline: small _Fake* classes
# exercise the seam without a runtime).
from cjm_graph_storage_adapter_interface.adapter import GraphStorageToolProtocol


class _FakeGraphTool:
    def __init__(self):
        self.nodes: Dict[str, GraphNode] = {}
        self.edges: Dict[str, GraphEdge] = {}
        self.seen_types: List[type] = []  # records what the adapter handed us

    def add_nodes(self, nodes):
        self.seen_types.extend(type(n) for n in nodes)
        for n in nodes:
            self.nodes[n.id] = n
        return [n.id for n in nodes]

    def add_edges(self, edges):
        self.seen_types.extend(type(e) for e in edges)
        for e in edges:
            self.edges[e.id] = e
        return [e.id for e in edges]

    def get_node(self, node_id):
        return self.nodes.get(node_id)

    def get_edge(self, edge_id):
        return self.edges.get(edge_id)

    def get_context(self, node_id, depth=1, filter_labels=None):
        return GraphContext(nodes=list(self.nodes.values()), edges=list(self.edges.values()),
                            metadata={"center": node_id, "depth": depth})

    def find_nodes_by_source(self, source_ref):
        self.seen_types.append(type(source_ref))
        return [n for n in self.nodes.values()
                if any(s.content_hash == source_ref.content_hash for s in n.sources)]

    def find_nodes_by_label(self, label, limit=100):
        return [n for n in self.nodes.values() if n.label == label][:limit]

    def query_nodes(self, query):
        self.seen_types.append(type(query))
        ns = [n for n in self.nodes.values() if query.label is None or n.label == query.label]
        if query.count:
            return NodeQueryResult(count=len(ns))
        return NodeQueryResult(nodes=ns)

    def query_edges(self, query):
        self.seen_types.append(type(query))
        if query.count:
            return EdgeQueryResult(count=len(self.edges))
        return EdgeQueryResult(edges=list(self.edges.values()))

    def raw_query(self, query):
        self.seen_types.append(type(query))
        assert query.backend == "fake", "backend mismatch must be refused by real tools"
        return RawQueryResult(columns=["one"], rows=[[1]], row_count=1, backend="fake")

    def update_node(self, node_id, properties):
        n = self.nodes.get(node_id)
        if n is None:
            return False
        n.properties.update(properties)
        return True

    def update_edge(self, edge_id, properties):
        e = self.edges.get(edge_id)
        if e is None:
            return False
        e.properties.update(properties)
        return True

    def delete_nodes(self, node_ids, cascade=True):
        count = 0
        for nid in node_ids:
            if self.nodes.pop(nid, None) is not None:
                count += 1
                if cascade:
                    self.edges = {eid: e for eid, e in self.edges.items()
                                  if e.source_id != nid and e.target_id != nid}
        return count

    def delete_edges(self, edge_ids):
        return sum(1 for eid in edge_ids if self.edges.pop(eid, None) is not None)

    def get_schema(self):
        return {"labels": sorted({n.label for n in self.nodes.values()})}

    def integrity_check(self):
        return {"ok": True, "errors": [], "backend": "fake"}

    def import_graph(self, graph_data, merge_strategy="overwrite"):
        self.add_nodes(graph_data.nodes)
        self.add_edges(graph_data.edges)
        return {"nodes": len(graph_data.nodes), "edges": len(graph_data.edges)}

    def export_graph(self, filter_query=None):
        return GraphContext(nodes=list(self.nodes.values()), edges=list(self.edges.values()))


tool = _FakeGraphTool()
assert isinstance(tool, GraphStorageToolProtocol)  # the structural contract holds

adapter = GenericGraphStorageAdapter(tool)

# wire dicts in -> the TOOL sees typed objects (the adapter is the typed boundary)
node_dicts = [GraphNode(id="n1", label="Segment", properties={"index": 0, "text": "hi"}).to_dict(),
              GraphNode(id="n2", label="Segment", properties={"index": 1, "text": ""}).to_dict()]
ids = adapter.add_nodes(node_dicts)
assert ids == ["n1", "n2"]
edge_ids = adapter.add_edges([GraphEdge(id="e1", source_id="n1", target_id="n2",
                                        relation_type="NEXT").to_dict()])
assert edge_ids == ["e1"]
assert all(t in (GraphNode, GraphEdge) for t in tool.seen_types)

# typed query via tagged wire dict -> typed result out
res = adapter.query_nodes(NodeQuery(label="Segment", count=True).to_dict())
assert isinstance(res, NodeQueryResult) and res.count == 2
assert tool.seen_types[-1] is NodeQuery

# typed objects pass through unchanged
res2 = adapter.query_edges(EdgeQuery(count=True))
assert isinstance(res2, EdgeQueryResult) and res2.count == 1

# raw escape: typed dict normalization + backend marking flows through
raw = adapter.raw_query(RawQuery(text="SELECT 1", backend="fake").to_dict())
assert isinstance(raw, RawQueryResult) and raw.backend == "fake"

# reverse index via wire dict
from cjm_context_graph_primitives.locators import FileRef
from cjm_context_graph_primitives.slices import CharSlice
ref = SourceRef(locator=FileRef(path="/runs/x.json"), content_hash="sha256:ab",
                slice=CharSlice(0, 2))
tool.nodes["n1"].sources.append(ref)
hits = adapter.find_nodes_by_source(ref.to_dict())
assert [n.id for n in hits] == ["n1"]

# import/export round-trip through normalization
exported = adapter.export_graph()
tool2 = _FakeGraphTool()
adapter2 = GenericGraphStorageAdapter(tool2)
counts = adapter2.import_graph(exported.to_dict())
assert counts == {"nodes": 2, "edges": 1}

# introspection
assert adapter.integrity_check()["ok"] is True
assert adapter.get_schema() == {"labels": ["Segment"]}

# update/delete forwarding
assert adapter.update_node("n1", {"text": "edited"}) is True
assert adapter.delete_edges(["e1"]) == 1
assert adapter.delete_nodes(["n1", "n2"]) == 2